# WM-811K weak margin 다중 seed 재현 실험

고정 split에서 baseline과 `weak_none_margin_005`를 seed 17, 42, 2026으로 쌍 비교합니다. 검증된 seed 42 결과는 재사용하며 Test 추론은 수행하지 않습니다.

In [ ]:
!pip install -q pandas==2.2.3 scikit-learn==1.6.1 matplotlib==3.10.0
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys, torch
drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임에서 T4 GPU를 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

파일을 하나씩 다섯 번 선택합니다. 기본 CNN, 기존 취약 후보, margin 후보, 다중 seed 실행기, split CSV 순서입니다.

In [ ]:
uploads = [
    ('train_wm811k_cnn.py', '1/5: train_wm811k_cnn.py를 선택하세요.'),
    ('train_wm811k_weak_class_candidates.py', '2/5: train_wm811k_weak_class_candidates.py를 선택하세요.'),
    ('train_wm811k_weak_margin_candidates.py', '3/5: train_wm811k_weak_margin_candidates.py를 선택하세요.'),
    ('run_wm811k_weak_margin_multiseed.py', '4/5: run_wm811k_weak_margin_multiseed.py를 선택하세요.'),
    ('split_assignments.csv', '5/5: split_assignments.csv를 선택하세요.'),
]
for target_name, message in uploads:
    print(message)
    uploaded = files.upload()
    assert len(uploaded) == 1, '한 번에 파일 하나만 선택하세요.'
    Path('/content', target_name).write_bytes(next(iter(uploaded.values())))
print('업로드 완료')

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS = DATA_DIR / 'wafer_maps_64.npy'
LABELS = DATA_DIR / 'labels.npy'
ASSIGNMENTS = Path('/content/split_assignments.csv')
SCRIPT = Path('/content/run_wm811k_weak_margin_multiseed.py')
SEED42_SOURCE = Path('/content/drive/MyDrive/AI_project/WM811K_weak_margin_experiment/validation_only_seed42')
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_weak_margin_multiseed')
required = [MAPS, LABELS, ASSIGNMENTS, SCRIPT, Path('/content/train_wm811k_cnn.py'), Path('/content/train_wm811k_weak_class_candidates.py'), Path('/content/train_wm811k_weak_margin_candidates.py'), SEED42_SOURCE / 'baseline' / 'run_summary.json', SEED42_SOURCE / 'weak_none_margin_005' / 'run_summary.json']
for path in required:
    assert path.is_file(), f'필수 파일이 없습니다: {path}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(SCRIPT), '--maps', str(MAPS), '--labels', str(LABELS), '--assignments', str(ASSIGNMENTS), '--output-dir', str(OUTPUT_DIR), '--seed42-source', str(SEED42_SOURCE), '--seeds', '17', '42', '2026', '--epochs', '20', '--batch-size', '256', '--num-workers', '2']
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'다중 seed 실행 실패: 종료 코드 {return_code}'
print('Validation-only 다중 seed 비교 완료:', OUTPUT_DIR)

In [ ]:
import json, pandas as pd
summary = json.loads((OUTPUT_DIR / 'reproducibility_summary.json').read_text(encoding='utf-8'))
display(summary)
display(pd.read_csv(OUTPUT_DIR / 'paired_seed_deltas.csv'))
display(pd.read_csv(OUTPUT_DIR / 'metric_summary.csv'))

In [ ]:
ZIP_BASE = Path('/content/wm811k_weak_margin_multiseed_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('다운로드 준비:', zip_path, f'{zip_path.stat().st_size / 1024**2:.2f} MB')
files.download(str(zip_path))

다운로드한 `wm811k_weak_margin_multiseed_results.zip`을 로컬 프로젝트의 `결과물/wm811k/colab_가져오기/`에 압축을 풀지 않고 넣으세요. 중단돼도 같은 셀을 다시 실행하면 완료된 seed·전략 결과를 재사용합니다.